In [ ]:
# Required Libraries
import json
import hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, asdict, field
import pickle

print("✅ Libraries loaded")

## 1. Basic Key-Value Memory Store

In [ ]:
class MemoryStore:
    """
    Simple in-memory key-value store with persistence.
    """
    
    def __init__(self, filepath: str = "memory_store.json"):
        self.filepath = Path(filepath)
        self.store: Dict[str, Any] = {}
        self.metadata: Dict[str, Dict] = {}
        self._load()
    
    def _load(self):
        """Load store from disk if exists."""
        if self.filepath.exists():
            with open(self.filepath, 'r') as f:
                data = json.load(f)
                self.store = data.get('store', {})
                self.metadata = data.get('metadata', {})
            print(f"📂 Loaded {len(self.store)} items from {self.filepath}")
    
    def _save(self):
        """Save store to disk."""
        self.filepath.parent.mkdir(parents=True, exist_ok=True)
        with open(self.filepath, 'w') as f:
            json.dump({
                'store': self.store,
                'metadata': self.metadata
            }, f, indent=2, default=str)
    
    def set(self, key: str, value: Any, tags: List[str] = None):
        """Store a value with optional tags."""
        self.store[key] = value
        self.metadata[key] = {
            'created_at': datetime.now().isoformat(),
            'updated_at': datetime.now().isoformat(),
            'tags': tags or [],
            'type': type(value).__name__
        }
        self._save()
        print(f"💾 Stored: {key}")
    
    def get(self, key: str, default: Any = None) -> Any:
        """Retrieve a value."""
        return self.store.get(key, default)
    
    def delete(self, key: str) -> bool:
        """Delete a key."""
        if key in self.store:
            del self.store[key]
            del self.metadata[key]
            self._save()
            print(f"🗑️ Deleted: {key}")
            return True
        return False
    
    def list_keys(self, tag: str = None) -> List[str]:
        """List all keys, optionally filtered by tag."""
        if tag:
            return [k for k, m in self.metadata.items() if tag in m.get('tags', [])]
        return list(self.store.keys())
    
    def clear(self):
        """Clear all data."""
        self.store.clear()
        self.metadata.clear()
        self._save()
        print("🧹 Memory cleared")
    
    def info(self) -> Dict:
        """Get store information."""
        return {
            'total_keys': len(self.store),
            'filepath': str(self.filepath),
            'keys': list(self.store.keys())[:10]  # First 10
        }

In [ ]:
# Create memory store
memory = MemoryStore("demo_memory.json")

# Store some values
memory.set("dataset_info", {
    "name": "titanic",
    "rows": 891,
    "columns": 12
}, tags=["dataset", "titanic"])

memory.set("model_params", {
    "model_type": "RandomForest",
    "n_estimators": 100
}, tags=["model", "params"])

In [ ]:
# Retrieve values
print("📊 Dataset Info:", memory.get("dataset_info"))
print("\n📋 All Keys:", memory.list_keys())
print("\n🏷️ Keys with 'model' tag:", memory.list_keys(tag="model"))

## 2. Session-Based Memory

In [ ]:
@dataclass
class Session:
    """
    Represents a session with its data.
    """
    session_id: str
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    data: Dict[str, Any] = field(default_factory=dict)
    history: List[Dict] = field(default_factory=list)
    
    def add_to_history(self, action: str, details: Dict = None):
        """Add an action to session history."""
        self.history.append({
            'timestamp': datetime.now().isoformat(),
            'action': action,
            'details': details or {}
        })

In [ ]:
class SessionManager:
    """
    Manage multiple sessions with persistence.
    """
    
    def __init__(self, storage_dir: str = "sessions"):
        self.storage_dir = Path(storage_dir)
        self.storage_dir.mkdir(parents=True, exist_ok=True)
        self.sessions: Dict[str, Session] = {}
        self.current_session_id: Optional[str] = None
    
    def _generate_session_id(self) -> str:
        """Generate a unique session ID."""
        timestamp = datetime.now().isoformat()
        return hashlib.md5(timestamp.encode()).hexdigest()[:12]
    
    def create_session(self, session_id: str = None) -> Session:
        """Create a new session."""
        session_id = session_id or self._generate_session_id()
        session = Session(session_id=session_id)
        self.sessions[session_id] = session
        self.current_session_id = session_id
        print(f"🆕 Created session: {session_id}")
        return session
    
    def get_session(self, session_id: str = None) -> Optional[Session]:
        """Get a session by ID or current session."""
        session_id = session_id or self.current_session_id
        return self.sessions.get(session_id)
    
    def set_data(self, key: str, value: Any, session_id: str = None):
        """Set data in a session."""
        session = self.get_session(session_id)
        if session:
            session.data[key] = value
            session.add_to_history('set_data', {'key': key})
    
    def get_data(self, key: str, session_id: str = None) -> Any:
        """Get data from a session."""
        session = self.get_session(session_id)
        return session.data.get(key) if session else None
    
    def save_session(self, session_id: str = None):
        """Save session to disk."""
        session = self.get_session(session_id)
        if session:
            filepath = self.storage_dir / f"{session.session_id}.json"
            with open(filepath, 'w') as f:
                json.dump(asdict(session), f, indent=2, default=str)
            print(f"💾 Saved session: {session.session_id}")
    
    def load_session(self, session_id: str) -> Optional[Session]:
        """Load session from disk."""
        filepath = self.storage_dir / f"{session_id}.json"
        if filepath.exists():
            with open(filepath, 'r') as f:
                data = json.load(f)
            session = Session(**data)
            self.sessions[session_id] = session
            print(f"📂 Loaded session: {session_id}")
            return session
        return None
    
    def list_sessions(self) -> List[str]:
        """List all session IDs."""
        # Memory + disk
        disk_sessions = [f.stem for f in self.storage_dir.glob("*.json")]
        memory_sessions = list(self.sessions.keys())
        return list(set(disk_sessions + memory_sessions))

In [ ]:
# Create session manager
session_mgr = SessionManager("demo_sessions")

# Create a new session
session = session_mgr.create_session("demo-001")

# Store session data
session_mgr.set_data("dataset", "titanic")
session_mgr.set_data("target", "Survived")
session_mgr.set_data("features", ["Pclass", "Age", "Sex"])

In [ ]:
# Get session data
print("📊 Session Data:")
print(f"   Dataset: {session_mgr.get_data('dataset')}")
print(f"   Target: {session_mgr.get_data('target')}")
print(f"   Features: {session_mgr.get_data('features')}")

print("\n📜 Session History:")
for h in session.history:
    print(f"   [{h['timestamp'][:19]}] {h['action']}: {h['details']}")

In [ ]:
# Save session
session_mgr.save_session()
print(f"\n📋 Available sessions: {session_mgr.list_sessions()}")

## 3. Conversation Memory

In [ ]:
@dataclass
class Message:
    """
    Represents a conversation message.
    """
    role: str  # 'user', 'assistant', 'system'
    content: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    metadata: Dict = field(default_factory=dict)


class ConversationMemory:
    """
    Memory for conversation history with context management.
    """
    
    def __init__(self, max_messages: int = 100):
        self.messages: List[Message] = []
        self.max_messages = max_messages
        self.context: Dict[str, Any] = {}
    
    def add_message(self, role: str, content: str, metadata: Dict = None):
        """Add a message to history."""
        message = Message(
            role=role,
            content=content,
            metadata=metadata or {}
        )
        self.messages.append(message)
        
        # Trim if exceeds max
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]
    
    def add_user_message(self, content: str):
        """Add a user message."""
        self.add_message('user', content)
    
    def add_assistant_message(self, content: str):
        """Add an assistant message."""
        self.add_message('assistant', content)
    
    def add_system_message(self, content: str):
        """Add a system message."""
        self.add_message('system', content)
    
    def get_history(self, n: int = None) -> List[Dict]:
        """Get message history."""
        messages = self.messages[-n:] if n else self.messages
        return [asdict(m) for m in messages]
    
    def get_formatted_history(self, n: int = None) -> str:
        """Get formatted conversation history."""
        messages = self.messages[-n:] if n else self.messages
        formatted = []
        for m in messages:
            formatted.append(f"[{m.role.upper()}]: {m.content}")
        return "\n".join(formatted)
    
    def set_context(self, key: str, value: Any):
        """Set conversation context."""
        self.context[key] = value
    
    def get_context(self, key: str, default: Any = None) -> Any:
        """Get conversation context."""
        return self.context.get(key, default)
    
    def clear(self):
        """Clear conversation history."""
        self.messages.clear()
        self.context.clear()
    
    def summary(self) -> Dict:
        """Get conversation summary."""
        return {
            'total_messages': len(self.messages),
            'user_messages': len([m for m in self.messages if m.role == 'user']),
            'assistant_messages': len([m for m in self.messages if m.role == 'assistant']),
            'context_keys': list(self.context.keys())
        }

In [ ]:
# Create conversation memory
conv_memory = ConversationMemory()

# Set context
conv_memory.set_context('dataset', 'titanic')
conv_memory.set_context('task', 'classification')

# Simulate conversation
conv_memory.add_system_message("You are a data science assistant.")
conv_memory.add_user_message("Load the titanic dataset")
conv_memory.add_assistant_message("I've loaded the Titanic dataset with 891 rows and 12 columns.")
conv_memory.add_user_message("What are the missing values?")
conv_memory.add_assistant_message("The dataset has missing values in Age (177), Cabin (687), and Embarked (2) columns.")
conv_memory.add_user_message("Train a classification model")
conv_memory.add_assistant_message("I'll train a Random Forest classifier with target='Survived'.")

In [ ]:
# View formatted history
print("📜 Conversation History:")
print(conv_memory.get_formatted_history())

In [ ]:
# Get summary
print("\n📊 Conversation Summary:")
for key, value in conv_memory.summary().items():
    print(f"   {key}: {value}")

## 4. Vector Store for Semantic Search

In [ ]:
import numpy as np
from collections import defaultdict


class SimpleVectorStore:
    """
    Simple vector store for semantic search using TF-IDF-like similarity.
    (For production, use ChromaDB, FAISS, or similar)
    """
    
    def __init__(self):
        self.documents: List[Dict] = []
        self.vocabulary: Dict[str, int] = {}
        self.idf: Dict[str, float] = {}
    
    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization."""
        return text.lower().replace('.', ' ').replace(',', ' ').split()
    
    def _compute_tf(self, tokens: List[str]) -> Dict[str, float]:
        """Compute term frequency."""
        tf = defaultdict(int)
        for token in tokens:
            tf[token] += 1
        total = len(tokens)
        return {k: v / total for k, v in tf.items()}
    
    def _update_idf(self):
        """Update inverse document frequency."""
        doc_count = defaultdict(int)
        for doc in self.documents:
            tokens = set(self._tokenize(doc['text']))
            for token in tokens:
                doc_count[token] += 1
        
        n_docs = len(self.documents)
        self.idf = {k: np.log(n_docs / (1 + v)) for k, v in doc_count.items()}
    
    def _vectorize(self, text: str) -> np.ndarray:
        """Convert text to TF-IDF vector."""
        tokens = self._tokenize(text)
        tf = self._compute_tf(tokens)
        
        # Build vocabulary if needed
        for token in tf:
            if token not in self.vocabulary:
                self.vocabulary[token] = len(self.vocabulary)
        
        # Create vector
        vector = np.zeros(len(self.vocabulary))
        for token, freq in tf.items():
            if token in self.vocabulary:
                idx = self.vocabulary[token]
                idf_val = self.idf.get(token, 1.0)
                vector[idx] = freq * idf_val
        
        return vector
    
    def add(self, text: str, metadata: Dict = None):
        """Add a document to the store."""
        doc = {
            'id': len(self.documents),
            'text': text,
            'metadata': metadata or {},
            'added_at': datetime.now().isoformat()
        }
        self.documents.append(doc)
        self._update_idf()
        print(f"📄 Added document {doc['id']}")
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Search for similar documents."""
        if not self.documents:
            return []
        
        query_vector = self._vectorize(query)
        
        results = []
        for doc in self.documents:
            doc_vector = self._vectorize(doc['text'])
            
            # Ensure same size
            max_len = max(len(query_vector), len(doc_vector))
            qv = np.zeros(max_len)
            dv = np.zeros(max_len)
            qv[:len(query_vector)] = query_vector
            dv[:len(doc_vector)] = doc_vector
            
            # Cosine similarity
            norm_q = np.linalg.norm(qv)
            norm_d = np.linalg.norm(dv)
            if norm_q > 0 and norm_d > 0:
                similarity = np.dot(qv, dv) / (norm_q * norm_d)
            else:
                similarity = 0
            
            results.append({
                'id': doc['id'],
                'text': doc['text'],
                'metadata': doc['metadata'],
                'score': float(similarity)
            })
        
        # Sort by similarity
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:top_k]

In [ ]:
# Create vector store
vector_store = SimpleVectorStore()

# Add documents
vector_store.add("Load the titanic dataset from CSV file", {"type": "data_loading"})
vector_store.add("Perform exploratory data analysis on the dataset", {"type": "eda"})
vector_store.add("Handle missing values by imputation", {"type": "preprocessing"})
vector_store.add("Train a random forest classification model", {"type": "modeling"})
vector_store.add("Evaluate model accuracy and confusion matrix", {"type": "evaluation"})
vector_store.add("Generate a report with visualizations", {"type": "reporting"})
vector_store.add("Feature engineering and selection", {"type": "preprocessing"})
vector_store.add("Hyperparameter tuning with cross validation", {"type": "optimization"})

In [ ]:
# Search for similar documents
query = "train a machine learning model"
results = vector_store.search(query, top_k=3)

print(f"🔍 Query: '{query}'")
print("\n📊 Results:")
for r in results:
    print(f"   [{r['score']:.3f}] {r['text']} ({r['metadata'].get('type', 'N/A')})")

In [ ]:
# Another search
query = "handle missing data"
results = vector_store.search(query, top_k=3)

print(f"🔍 Query: '{query}'")
print("\n📊 Results:")
for r in results:
    print(f"   [{r['score']:.3f}] {r['text']}")

## 5. Combined Memory System

In [ ]:
class AgentMemory:
    """
    Combined memory system for agents with multiple storage types.
    """
    
    def __init__(self, session_id: str = None):
        self.session_id = session_id or datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Different memory types
        self.kv_store = {}  # Key-value store
        self.conversation = ConversationMemory()
        self.vector_store = SimpleVectorStore()
        
        # Artifacts storage
        self.artifacts: Dict[str, Any] = {}
        
        print(f"🧠 Agent Memory initialized - Session: {self.session_id}")
    
    # Key-Value operations
    def store(self, key: str, value: Any):
        """Store a key-value pair."""
        self.kv_store[key] = {
            'value': value,
            'timestamp': datetime.now().isoformat()
        }
    
    def retrieve(self, key: str, default: Any = None) -> Any:
        """Retrieve a value."""
        item = self.kv_store.get(key)
        return item['value'] if item else default
    
    # Conversation operations
    def add_exchange(self, user_msg: str, assistant_msg: str):
        """Add a conversation exchange."""
        self.conversation.add_user_message(user_msg)
        self.conversation.add_assistant_message(assistant_msg)
    
    def get_recent_context(self, n: int = 5) -> str:
        """Get recent conversation context."""
        return self.conversation.get_formatted_history(n)
    
    # Vector search operations
    def remember(self, text: str, metadata: Dict = None):
        """Add to semantic memory."""
        self.vector_store.add(text, metadata)
    
    def recall(self, query: str, n: int = 3) -> List[Dict]:
        """Search semantic memory."""
        return self.vector_store.search(query, n)
    
    # Artifact management
    def save_artifact(self, name: str, artifact: Any, artifact_type: str = "object"):
        """Save an artifact (model, dataframe, etc.)."""
        self.artifacts[name] = {
            'artifact': artifact,
            'type': artifact_type,
            'saved_at': datetime.now().isoformat()
        }
        print(f"📦 Saved artifact: {name} ({artifact_type})")
    
    def get_artifact(self, name: str) -> Any:
        """Get an artifact."""
        item = self.artifacts.get(name)
        return item['artifact'] if item else None
    
    def list_artifacts(self) -> List[Dict]:
        """List all artifacts."""
        return [
            {'name': k, 'type': v['type'], 'saved_at': v['saved_at']}
            for k, v in self.artifacts.items()
        ]
    
    def summary(self) -> Dict:
        """Get memory summary."""
        return {
            'session_id': self.session_id,
            'kv_items': len(self.kv_store),
            'messages': len(self.conversation.messages),
            'semantic_docs': len(self.vector_store.documents),
            'artifacts': len(self.artifacts)
        }

In [ ]:
# Create agent memory
agent_memory = AgentMemory("demo-session")

# Store key-value data
agent_memory.store('dataset_name', 'titanic')
agent_memory.store('target_column', 'Survived')

# Add conversation
agent_memory.add_exchange(
    "Load the titanic dataset",
    "Dataset loaded successfully with 891 rows and 12 columns."
)
agent_memory.add_exchange(
    "Train a model",
    "Trained Random Forest with 85% accuracy."
)

# Add semantic memories
agent_memory.remember("Loaded titanic.csv dataset", {"action": "load"})
agent_memory.remember("Trained Random Forest classifier", {"action": "train"})
agent_memory.remember("Generated accuracy report", {"action": "report"})

In [ ]:
# Get memory summary
print("🧠 Memory Summary:")
for key, value in agent_memory.summary().items():
    print(f"   {key}: {value}")

In [ ]:
# Retrieve stored data
print(f"📊 Dataset: {agent_memory.retrieve('dataset_name')}")
print(f"🎯 Target: {agent_memory.retrieve('target_column')}")

# Get conversation context
print("\n📜 Recent Context:")
print(agent_memory.get_recent_context())

In [ ]:
# Semantic search
print("\n🔍 Recall 'training model':")
for r in agent_memory.recall("training model"):
    print(f"   [{r['score']:.2f}] {r['text']}")

## ✅ Summary

This memory module provides:

**1. MemoryStore**
- Key-value storage with persistence
- Tagging support
- Metadata tracking

**2. SessionManager**
- Session-based data organization
- History tracking
- Session persistence

**3. ConversationMemory**
- Message history
- Context management
- Formatted output

**4. SimpleVectorStore**
- TF-IDF based similarity
- Semantic search
- Document indexing

**5. AgentMemory**
- Combined memory system
- Multiple storage types
- Artifact management